# Credit Card Fraud Detection - Machine Learning Pipeline

This notebook demonstrates the complete end-to-end Machine Learning pipeline for credit card fraud detection:
1. **Data Loading & Schema Validation**
2. **Train/Test Splitting** (Preventing Data Leakage)
3. **Standard Scaling** (Applied to Time and Amount)
4. **Class Imbalance Handling with SMOTE** (Applied strictly on Training Split)
5. **Model Training & Comparison** (Logistic Regression, Random Forest, XGBoost)
6. **Model Evaluation** (Precision, Recall, F1, ROC-AUC, PR-AUC, Confusion Matrix)
7. **Model Artifact Export** (`best_model.joblib` & `scaler.joblib`)

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import prepare_train_test_data
from src.evaluation import evaluate_predictions, plot_confusion_matrix_fig, plot_metrics_comparison_fig
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

## 1. Load Data & Prepare Train/Test Splits with SMOTE

In [ ]:
X_train_res, y_train_res, X_test_scaled, y_test, scaler, feature_cols = prepare_train_test_data(
    data_path="../data/creditcard.csv", test_size=0.2, random_state=42
)

## 2. Train Models

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

results = {}
for name, model in classifiers.items():
    print(f"Training {name}...")
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = evaluate_predictions(y_test, y_pred, y_prob)

## 3. Comparative Evaluation

In [ ]:
results_df = pd.DataFrame(results).T
results_df[['precision', 'recall', 'f1', 'roc_auc', 'pr_auc']]